# E-Commerce Sales Analytics
**Dataset:** Brazilian E-Commerce (Olist) via Kaggle  
**Author:** Fikri Firstly Arrasyid Hawe  
**Goal:** Analyze 100K+ orders to uncover revenue patterns, top categories, and delivery performance.

---
### Setup
Run `pip install kagglehub pandas matplotlib seaborn` before starting.

In [ ]:
import kagglehub
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print('Files:', os.listdir(path))

orders = pd.read_csv(os.path.join(path, 'olist_orders_dataset.csv'))
order_items = pd.read_csv(os.path.join(path, 'olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(path, 'olist_products_dataset.csv'))
category_trans = pd.read_csv(os.path.join(path, 'product_category_name_translation.csv'))
payments = pd.read_csv(os.path.join(path, 'olist_order_payments_dataset.csv'))

print(f'Orders: {len(orders):,} rows')

## 1. Data Overview & Cleaning

In [ ]:
# Parse dates
for col in ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col])

# Only delivered orders
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered['delivery_days'] = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.days
delivered['late'] = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']
print(f'Delivered orders: {len(delivered):,}')
print(f'Late deliveries: {delivered["late"].sum():,} ({delivered["late"].mean()*100:.1f}%)')

## 2. Revenue by Category

In [ ]:
# Merge products with English category names
products_en = products.merge(category_trans, on='product_category_name', how='left')
items_rich = order_items.merge(products_en[['product_id', 'product_category_name_english']], on='product_id', how='left')

# Revenue per category
cat_revenue = (items_rich.groupby('product_category_name_english')['price']
               .sum().sort_values(ascending=False).head(15))

cat_revenue.plot(kind='barh', color='#c8a87a', figsize=(10, 7))
plt.title('Top 15 Categories by Revenue (BRL)', fontsize=13)
plt.xlabel('Total Revenue (BRL)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Monthly Revenue Trend

In [ ]:
payments_agg = payments.groupby('order_id')['payment_value'].sum().reset_index()
orders_pay = delivered.merge(payments_agg, on='order_id', how='left')
orders_pay['month'] = orders_pay['order_purchase_timestamp'].dt.to_period('M')

monthly = orders_pay.groupby('month')['payment_value'].sum()
monthly.index = monthly.index.astype(str)

plt.figure(figsize=(14, 5))
plt.plot(monthly.index, monthly.values, color='#c8a87a', linewidth=2.5, marker='o', markersize=4)
plt.fill_between(range(len(monthly)), monthly.values, alpha=0.1, color='#c8a87a')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Revenue Trend', fontsize=13)
plt.ylabel('Revenue (BRL)')
plt.tight_layout()
plt.show()

## 4. Delivery Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

delivered['delivery_days'].clip(0, 60).hist(bins=40, color='#c8a87a', edgecolor='white', ax=axes[0])
axes[0].set_title('Delivery Time Distribution (days)')
axes[0].set_xlabel('Days')

late_data = delivered.groupby(delivered['order_purchase_timestamp'].dt.to_period('M'))['late'].mean() * 100
late_data.index = late_data.index.astype(str)
axes[1].bar(range(len(late_data)), late_data.values, color='#1a1a1a', alpha=0.7)
axes[1].set_title('Late Delivery Rate by Month (%)')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

## 5. Conclusions

- **Health & beauty, watches & gifts, bed & bath** are the top revenue-generating categories
- Monthly revenue showed strong growth through 2017, with a dip in late 2018
- Average delivery time is **~12 days**; late deliveries affect ~7.5% of orders
- **Recommendation:** Prioritize logistics optimization in high-volume months (Nov-Dec) to reduce late delivery rates